# 5 — Évaluation et sélection du modèle

Ce notebook agrège les résultats des quatre notebooks de modélisation (`3a` à `3d`), produit la **comparaison inter-familles** et désigne le modèle champion qui sera utilisé pour la soumission Kaggle (NB5).

**Pré-requis** : avoir exécuté `3a_scaled_models.ipynb`, `3b_sklearn_trees.ipynb`, `3c_native_boosting.ipynb`, `3d_stacking.ipynb` au moins une fois — chacun écrit son fichier `results/family_<nom>.json`.

In [1]:
%load_ext autoreload
%autoreload 2
%run 2_data_prep.ipynb

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_DIR = Path('results')

Dimensions brutes : (1460, 81)
Nombre de points supprimés : 2
Dimensions après suppression : (1458, 81)
Variables après ingénierie : 90 colonnes (+8 dérivées)
Corrélation des variables dérivées avec SalePrice_log :
  TotalSF            : +0.825
  TotalBathrooms     : +0.677
  HouseAge           : -0.588
  YearsSinceRemodel  : -0.569
  GarageAge          : -0.543
  HasGarage          : +0.323
  HasSecondFloor     : +0.151
  HasPool            : +0.077   <-- faible (|r| < 0.1)
X_train : (1166, 83)   |   X_test : (292, 83)
Colonnes retirées (colinéarité) : ['GarageArea', 'TotalBsmtSF', 'TotRmsAbvGrd', 'GarageYrBlt']
NaN dans X_train : 6276 cellules sur 96778
Audit de cardinalité des colonnes nominales (sur X_train) :
Colonne           modalités   bucket
  Neighborhood           25     TargetEncoder (haute)  [imput. mode]
  Exterior2nd            16     TargetEncoder (haute)  [imput. mode]
  Exterior1st            15     TargetEncoder (haute)  [imput. mode]
  MSSubClass             15     

## 5.1 Agrégation des résultats

In [2]:
result_files = sorted(RESULTS_DIR.glob('family_*.json'))
print(f"Fichiers trouvés : {len(result_files)}")
for p in result_files:
    print(f"  - {p.name}")

if not result_files:
    raise RuntimeError("Aucun fichier results/family_*.json trouvé. "
                       "Exécuter 3a/3b/3c/3d d'abord.")

all_results = []
for p in result_files:
    all_results.extend(json.loads(p.read_text()))

results_df = pd.DataFrame(all_results).sort_values('holdout_rmsle').reset_index(drop=True)
print(f"\nNombre total de modèles agrégés : {len(results_df)}")
display(results_df[['family', 'model', 'cv_rmsle', 'holdout_rmsle', 'fit_time_s']])

Fichiers trouvés : 4
  - family_native_boosting.json
  - family_scaled.json
  - family_sklearn_trees.json
  - family_stacking.json

Nombre total de modèles agrégés : 17


,family,model,cv_rmsle,holdout_rmsle,fit_time_s
0,stacking,Stacking,NaN,0.112202,14.122129
1,sklearn_trees,GradientBoosting,0.125168,0.115419,2.757908
2,scaled,SVR,0.117750,0.115885,3.272943
3,scaled,Lasso,0.119997,0.116451,0.104538
4,scaled,OLS,0.121470,0.116999,0.032703
5,native_boosting,CatBoost,0.122280,0.117485,7.778354
6,native_boosting,XGBoost_onehot,0.123717,0.117796,1.490634
7,scaled,ElasticNet,0.119578,0.118910,0.388632
8,native_boosting,XGBoost_tuned,0.117110,0.119286,1.032960
9,native_boosting,XGBoost_native,0.125556,0.121032,1.772608


## 5.2 Comparaison inter-familles (RMSLE holdout)

C'est le graphique central de la Phase 5 : tous les modèles, toutes familles confondues, classés par performance sur le jeu de test.

In [3]:
palette = {'scaled': '#3498db', 'sklearn_trees': '#2ecc71', 'native_boosting': '#e67e22', 'stacking': '#9b59b6'}
colors = [palette.get(f, '#888') for f in results_df['family']]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(results_df)), results_df['holdout_rmsle'], color=colors)
ax.set_yticks(range(len(results_df)))
ax.set_yticklabels([f"{r['model']}  ({r['family']})" for _, r in results_df.iterrows()])
ax.invert_yaxis()
ax.set_xlabel('RMSLE (holdout) — plus bas = meilleur')
ax.set_title("Inved Corp — classement inter-familles par RMSLE")

for bar, v in zip(bars, results_df['holdout_rmsle']):
    ax.text(v + 0.001, bar.get_y() + bar.get_height()/2, f"{v:.4f}", va='center', fontweight='bold', fontsize=9)

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in palette.values()]
ax.legend(handles, palette.keys(), title='Famille', loc='lower right')
plt.tight_layout()
plt.show()

/tmp/ipykernel_234533/461718519.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5.2.1 Le classement est-il statistiquement fiable ? (bootstrap)

Le holdout ne compte que ~292 biens : un écart de RMSLE de quelques millièmes peut n'être que du **bruit d'échantillonnage**. Pour le quantifier, on **rééchantillonne** le holdout avec remise (2000 tirages, graine fixe) et on recalcule la RMSLE à chaque fois → un **intervalle de confiance à 95 %** par modèle. Si les IC des premiers se chevauchent, le classement de tête est un **quasi ex-æquo statistique**, et désigner « le » meilleur sur la seule 4ᵉ décimale serait abusif.

In [4]:
# Prédictions holdout (réutilisées par §5.5) + bootstrap des résidus
preds, y_true = {}, None
for pf in sorted(RESULTS_DIR.glob('preds_*.npz')):
    d = np.load(pf)
    for k in d.files:
        if k == 'y_true':
            y_true = d[k]
        else:
            preds[k] = d[k]


def bootstrap_rmsle_ci(y_t, y_p, n_boot=2000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n = len(y_t)
    stats = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        stats[b] = np.sqrt(np.mean((y_t[idx] - y_p[idx]) ** 2))
    return np.percentile(stats, 2.5), np.percentile(stats, 97.5)

ci = {m: bootstrap_rmsle_ci(y_true, preds[m]) for m in preds}
ci_df = (results_df[results_df['model'].isin(ci)]
         .assign(ci_low=lambda d: d['model'].map(lambda m: ci[m][0]),
                 ci_high=lambda d: d['model'].map(lambda m: ci[m][1]))
         .sort_values('holdout_rmsle').reset_index(drop=True))

fig, ax = plt.subplots(figsize=(11, 6))
yloc = range(len(ci_df))
err = np.array([ci_df['holdout_rmsle'] - ci_df['ci_low'], ci_df['ci_high'] - ci_df['holdout_rmsle']])
ax.errorbar(ci_df['holdout_rmsle'], yloc, xerr=err, fmt='o', color='#34495e',
            ecolor='#aaa', capsize=4)
ax.set_yticks(list(yloc))
ax.set_yticklabels([f"{r['model']} ({r['family']})" for _, r in ci_df.iterrows()])
ax.invert_yaxis()
champ_hi = ci_df.iloc[0]['ci_high']
ax.axvline(champ_hi, color='crimson', ls='--', lw=1, label=f"borne haute IC du champion ({champ_hi:.4f})")
ax.set_xlabel('RMSLE holdout (IC 95 % bootstrap)')
ax.set_title("Classement avec incertitude — les barres qui chevauchent = ex-æquo statistique")
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

tie = ci_df[ci_df['ci_low'] <= champ_hi]
print(f"Champion : {ci_df.iloc[0]['model']}  RMSLE {ci_df.iloc[0]['holdout_rmsle']:.4f} "
      f"[IC95 {ci_df.iloc[0]['ci_low']:.4f}–{ci_df.iloc[0]['ci_high']:.4f}]")
print(f"Modèles dont l'IC chevauche celui du champion (quasi ex-æquo) : "
      f"{', '.join(tie['model'].tolist())}")

Champion : Stacking  RMSLE 0.1122 [IC95 0.0963–0.1297]
Modèles dont l'IC chevauche celui du champion (quasi ex-æquo) : Stacking, GradientBoosting, SVR, Lasso, OLS, CatBoost, XGBoost_onehot, ElasticNet, XGBoost_tuned, XGBoost_native, Ridge, MLPRegressor, LightGBM, RandomForest


/tmp/ipykernel_234533/94877750.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5.3 CV RMSLE vs Holdout RMSLE — vérification de cohérence

Un grand écart entre CV et holdout pour un modèle signale un risque de surapprentissage (CV optimiste) ou de chance du split (holdout chanceux). Idéalement, les deux métriques sont proches : c'est le signe d'un modèle qui *généralise* bien.

In [5]:
mask = results_df['cv_rmsle'].notna()
fig, ax = plt.subplots(figsize=(8, 7))
sub = results_df[mask]
ax.scatter(sub['cv_rmsle'], sub['holdout_rmsle'],
           c=[palette.get(f, '#888') for f in sub['family']], s=120, edgecolor='black')
for _, r in sub.iterrows():
    ax.annotate(r['model'], (r['cv_rmsle'], r['holdout_rmsle']),
                xytext=(5, 5), textcoords='offset points', fontsize=9)
lims = [min(sub['cv_rmsle'].min(), sub['holdout_rmsle'].min()) * 0.95,
        max(sub['cv_rmsle'].max(), sub['holdout_rmsle'].max()) * 1.05]
ax.plot(lims, lims, 'r--', alpha=0.5, label='CV = Holdout')
ax.set_xlabel('CV RMSLE (5-fold)')
ax.set_ylabel('Holdout RMSLE')
ax.set_title("Cohérence CV ↔ Holdout par modèle")
ax.legend()
plt.tight_layout()
plt.show()

/tmp/ipykernel_234533/2905347478.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lecture — les deux classements ne s'accordent pas, et c'est instructif.**

- **GradientBoosting** est 2ᵉ sur le holdout (0,1154) mais ~8ᵉ en CV (0,1252) : son holdout est **chanceux** (un split favorable), pas le signe d'une supériorité robuste.
- **XGBoost tuné** est au contraire **meilleur en CV** (0,1171, optimisé par Optuna *sur la CV*) mais retombe ~9ᵉ sur le holdout — preuve que le holdout seul est un juge bruité.
- Le champion **Stacking est absent de ce graphique** : `StackingRegressor` ne renvoie pas de `cv_rmsle` ici (quirk de CV emboîtée, publié à `None`). On ne le force pas ; sa fiabilité est attestée autrement — par l'**IC bootstrap** du §5.2.1 (sur le holdout) et par la **stabilité CV ↔ holdout de ses base learners** (Lasso, GBR, XGBoost), tous proches de la diagonale ci-dessus.

Conclusion méthodologique : **ne jamais trancher sur le seul holdout**. Le choix du champion (§5.5) s'appuiera donc sur l'IC bootstrap *et* la cohérence CV, pas sur la 4ᵉ décimale d'une métrique unique.

## 5.4 Désignation du champion

In [6]:
winner = results_df.iloc[0].to_dict()
print(f"Champion : {winner['model']}   (famille : {winner['family']})")
print(f"   CV RMSLE      : {winner['cv_rmsle']}")
print(f"   Holdout RMSLE : {winner['holdout_rmsle']:.4f}")
print(f"   fit_time      : {winner['fit_time_s']:.2f}s")
print(f"   params        : {winner.get('params')}")

(RESULTS_DIR / 'winning_model.json').write_text(json.dumps(winner, indent=2, default=str))
print(f"\nÉcrit : {RESULTS_DIR / 'winning_model.json'}")

Champion : Stacking   (famille : stacking)
   CV RMSLE      : nan
   Holdout RMSLE : 0.1122
   fit_time      : 14.12s
   params        : {'meta': 'RidgeCV', 'base': ['Lasso', 'GBR', 'XGB']}

Écrit : results/winning_model.json


## 5.5 Dissertation : *« qu'est-ce qu'un bon modèle ? »*

### 5.5.1 Trois axes, pas une métrique

Un bon modèle n'est pas « celui qui a la plus petite RMSLE » mais celui qui tient sur les **trois axes** de la Data Science :

1. **Mathématique** — performance *et* robustesse statistique : RMSLE, mais lue avec son incertitude (IC bootstrap §5.2.1), sa cohérence CV ↔ holdout (§5.3), et les hypothèses propres à chaque famille.
2. **Informatique (opérationnel)** — latence d'inférence *mesurée* vs le SLA < 5 s du ML Canvas, taille d'artefact, et surtout **surface de monitoring** (un modèle unique vs un ensemble de N sous-modèles à surveiller).
3. **Métier** — interprétabilité (la promesse « top-3 facteurs » du Canvas), gestion de l'**asymétrie des erreurs**, et **équité géographique**.

> Ces trois axes ne servent pas qu'ici : ils seront **re-suivis en continu en Phase 6** comme les trois **couches de monitoring** (mathématique / système / métier). La Phase 5 les utilise une fois, *statiquement*, pour **choisir** le champion ; la Phase 6 les rebranche, *en continu*, pour le **surveiller**.

**L'erreur n'est pas symétrique (cellule *Impact* du ML Canvas).** En dollars, le signe compte : **sur-estimer** (IA > marché) fait stagner le bien et rate le KPI « +20 % de ventes sous 90 jours » ; **sous-estimer** (IA < marché) vend vite mais perd commission et valeur client — et c'est le déclencheur du *conseil rénovation*. La RMSLE, relative et symétrique en log, ne distingue pas les deux : d'où (1) la **validation finale par un consultant** et (2) un suivi métier séparé (taux d'override, délai de vente) en Phase 6. Le choix du log reste juste métier — une erreur de 10 k\$ ne pèse pas pareil sur 100 k\$ et sur 1 M\$.

### 5.5.2 Axe mathématique — performance, robustesse et limites par famille

La robustesse statistique est traitée en amont : l'**IC bootstrap (§5.2.1)** est sans appel — l'intervalle du champion chevauche celui de **quatorze des dix-sept modèles**, seul le bas de tableau (KNN, AdaBoost, arbre seul) s'en détache nettement. Autrement dit, sur ~292 biens de holdout, l'écart de tête est un **quasi ex-æquo statistique**. Le **désaccord CV ↔ holdout (§5.3)** enfonce le clou (GradientBoosting « chanceux », XGBoost tuné meilleur en CV). On ne choisit donc surtout pas sur la 4ᵉ décimale.

Restent les **hypothèses propres à chaque famille** — les connaître, c'est savoir *quand* le modèle échouera :

- **Linéaires (OLS, Ridge, Lasso, ElasticNet).** Supposent la cible linéaire en `log1p(SalePrice)`, résidus homoscédastiques et ~normaux (diagnostics §4.1.1 : QQ-plot, résidus vs ajustés). Ne captent pas nativement les interactions. La régularisation stabilise sous colinéarité ; Lasso sélectionne (léger biais). Ils sont **remarquablement compétitifs** (Lasso ≈ 0,116) : le log + les agrégats (`TotalSF`, `TotalBathrooms`) ont « linéarisé » l'essentiel. Ridge décroche (≈ 0,124) — sa pénalité L2 ne supprime aucune variable et subit le bruit des colonnes faibles.
- **KNN.** Aucune hypothèse paramétrique mais dépendance totale à la **distance** : sensible à l'échelle (`preprocessor_scaled`) et à la **dimension** ; le plus faible à l'échelle (≈ 0,185).
- **Arbre seul.** Capture interactions/non-linéarités par seuils mais **sur-apprend** (variance) et **n'extrapole pas** hors domaine — un bien plus grand que tout l'historique est plafonné (courbe d'overfitting §4.2.1). Faible seul (≈ 0,18).
- **Ensembles d'arbres (RF, GradientBoosting, AdaBoost, XGBoost, LightGBM, CatBoost).** L'agrégation corrige la variance ; le **boosting domine le bagging** ici (GBR ≈ 0,115 vs RF ≈ 0,138). Limites : sensibilité aux hyperparamètres (Optuna §4.3.3 ; LightGBM non réglé sur-apprend à ≈ 0,131), opacité sans SHAP, **pas d'extrapolation**. CatBoost, meilleur booster isolé (≈ 0,117), illustre l'apport de l'*ordered boosting*.
- **Stacking.** Tire sa force de la **diversité des erreurs** (Lasso + GBR + XGBoost, trois biais inductifs) recombinée par un méta-RidgeCV. Risque : base learners trop corrélés → gain nul, voire méta-sur-apprentissage des prédictions *out-of-fold*. Gain réel mais **mince** (0,1122 vs 0,1154), à confronter aux axes suivants.

Les deux visuels ci-dessous appuient l'axe mathématique : la **grille prédit-vs-réel** (échecs partagés = problème de *données*, pas de modèle) et le **scatter en dollars** du champion (erreurs dans l'espace de décision du consultant).

#### 5.5.2.1 Grille prédit-vs-réel par modèle

Support visuel de l'axe mathématique : chaque vignette place les biens du holdout (prix réel en abscisse, prédit en ordonnée), **axes partagés** pour que « la distance à la diagonale » soit comparable d'une vignette à l'autre. Un point loin de la diagonale rouge = un bien mal prédit. Si les modèles ratent les **mêmes** biens (souvent les maisons atypiques / haut de gamme), l'erreur tient aux **données**, pas au modèle.

In [7]:
ordered = [m for m in results_df['model'] if m in preds]   # trié par RMSLE holdout
rmsle_by = dict(zip(results_df['model'], results_df['holdout_rmsle']))

ncols = 4
nrows = int(np.ceil(len(ordered) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.6 * nrows),
                         sharex=True, sharey=True)
axes = np.array(axes).reshape(-1)
lo, hi = float(y_true.min()), float(y_true.max())
for ax, m in zip(axes, ordered):
    ax.scatter(y_true, preds[m], alpha=0.3, s=10)
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1)
    ax.set_title(f"{m}\nRMSLE {rmsle_by.get(m, float('nan')):.4f}", fontsize=9)
    ax.set_xlabel('réel (log)', fontsize=8)
    ax.set_ylabel('prédit (log)', fontsize=8)
for ax in axes[len(ordered):]:
    ax.axis('off')
fig.suptitle("Grille prédit-vs-réel par modèle (holdout, axes partagés) — un point loin de la diagonale = bien mal prédit", fontsize=12)
plt.tight_layout()
plt.show()

/tmp/ipykernel_234533/3612402222.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


#### 5.5.2.2 Diagnostic du champion en dollars

On repasse le champion en **dollars** (`expm1`) — l'espace de décision du consultant — et on annote ses **3 plus grosses erreurs absolues** par leur `Id`, pour qu'un relecteur puisse inspecter ces biens (souvent atypiques) dans `data/train.csv`.

In [8]:
champion_name = winner['model']
assert champion_name in preds, f"Prédictions holdout absentes pour {champion_name}"
assert np.allclose(y_true, y_test_log.values), "Désalignement entre y_true (npz) et le split courant"

ids = df_cleaned.loc[X_test.index, 'Id'].values
y_pred_log = preds[champion_name]
dollars_true = np.expm1(y_true)
dollars_pred = np.expm1(y_pred_log)
abs_err = np.abs(dollars_pred - dollars_true)
worst = np.argsort(abs_err)[-3:][::-1]

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(dollars_true, dollars_pred, alpha=0.4, s=20)
lo, hi = float(dollars_true.min()), float(dollars_true.max())
ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='y = x')
for w in worst:
    ax.scatter(dollars_true[w], dollars_pred[w], color='crimson', s=60, zorder=5)
    ax.annotate(f"Id {int(ids[w])}", (dollars_true[w], dollars_pred[w]),
                xytext=(6, 6), textcoords='offset points', color='crimson', fontsize=9)
ax.set_xlabel('Prix réel ($)')
ax.set_ylabel('Prix prédit ($)')
ax.set_title(f"{champion_name} — prédit vs réel (espace dollars)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"3 plus grosses erreurs absolues — {champion_name} :")
for w in worst:
    print(f"  Id {int(ids[w]):4d} : réel {dollars_true[w]:>10,.0f} $  |  "
          f"prédit {dollars_pred[w]:>10,.0f} $  |  écart {abs_err[w]:>9,.0f} $")

3 plus grosses erreurs absolues — Stacking :
  Id  219 : réel    311,500 $  |  prédit    220,583 $  |  écart    90,917 $
  Id  262 : réel    276,000 $  |  prédit    340,870 $  |  écart    64,870 $
  Id 1045 : réel    278,000 $  |  prédit    341,969 $  |  écart    63,969 $


/tmp/ipykernel_234533/700298762.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5.5.3 Axe informatique — latence, artefact et surface de monitoring

Le coût opérationnel ne se mesure **pas** en temps d'*entraînement* (`fit_time_s`, hors-ligne et mensuel) — confusion fréquente — mais en **latence d'inférence** (tenue du SLA < 5 s du Canvas), en taille d'artefact, et surtout en **surface de monitoring** : un modèle unique a une seule chose à surveiller pour la dérive ; le Stacking en a **quatre** (3 base learners + méta), chacun pouvant dériver indépendamment.

On chronomètre le `predict` (et non le `fit`), et **au bon grain** : le SLA Canvas porte sur **une soumission de formulaire = un seul bien**. On mesure donc la **latence par requête** (predict sur 1 ligne) — le chiffre qui fait foi — car le coût fixe (mise en place des transformeurs) ne s'amortit pas sur une requête unitaire. On reporte aussi le **débit par lot** (ms/bien sur les 292 du holdout) à titre indicatif : il est plus optimiste car ce coût fixe s'y dilue.

In [9]:
import time as _time
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor
from xgboost import XGBRegressor

# Finalistes = champion + meilleurs représentants de chaque famille + un repère interprétable
FINALISTS = ['Stacking', 'GradientBoosting', 'CatBoost', 'XGBoost_tuned', 'Lasso']

_cat_cols = X_train.select_dtypes(include=['object', 'string']).columns.tolist()
_nb = {r['model']: r for r in json.loads((RESULTS_DIR / 'family_native_boosting.json').read_text())}
_xgb_keys = {'n_estimators', 'learning_rate', 'max_depth', 'min_child_weight',
             'subsample', 'colsample_bytree', 'reg_alpha', 'reg_lambda'}
_xgb_tuned_p = {k: v for k, v in (_nb['XGBoost_tuned']['params'] or {}).items() if k in _xgb_keys}

def _sc(m): return Pipeline([('preprocessor', preprocessor_scaled), ('model', m)])
def _en(m): return Pipeline([('preprocessor', preprocessor_encoded), ('model', m)])
def _na(m): return Pipeline([('preprocessor', preprocessor_native), ('model', m)])

def build_finalist(name):
    """Reconstruit un finaliste (mêmes configs que 3a–3d). Local à la Phase 5 — NB5 a son propre dispatcher."""
    if name == 'Lasso':
        return _sc(LassoCV(alphas=np.logspace(-4, 2, 20), cv=5, max_iter=50_000, random_state=RANDOM_STATE))
    if name == 'GradientBoosting':
        return _en(GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE))
    if name == 'CatBoost':
        return Pipeline([('preprocessor', CatBoostPrep()),
                         ('model', CatBoostRegressorCV(cat_features=_cat_cols, iterations=500,
                                                       learning_rate=0.05, depth=6, random_seed=RANDOM_STATE))])
    if name == 'XGBoost_tuned':
        return _na(XGBRegressor(**_xgb_tuned_p, enable_categorical=True, tree_method='hist',
                                random_state=RANDOM_STATE, n_jobs=-1, verbosity=0))
    if name == 'Stacking':
        return StackingRegressor(
            estimators=[('lasso', _sc(LassoCV(alphas=np.logspace(-4, 2, 20), cv=5, max_iter=50_000, random_state=RANDOM_STATE))),
                        ('gbr', _en(GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE))),
                        ('xgb', _na(XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, enable_categorical=True,
                                                 tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)))],
            final_estimator=RidgeCV(alphas=np.logspace(-2, 2, 10)), cv=5, n_jobs=-1)
    raise ValueError(name)

def _median_predict_ms(pipe, Xd, repeats):
    pipe.predict(Xd)  # warmup
    ts = []
    for _ in range(repeats):
        t0 = _time.perf_counter(); pipe.predict(Xd); ts.append(_time.perf_counter() - t0)
    return float(np.median(ts)) * 1000  # ms

X_one = X_test.iloc[[0]]   # une soumission de formulaire = un seul bien (grain du SLA)
fitted_finalists, latency_req_ms, latency_batch_ms_row = {}, {}, {}
for name in FINALISTS:
    pipe = build_finalist(name)
    pipe.fit(X_train, y_train_log)
    fitted_finalists[name] = pipe
    latency_req_ms[name] = _median_predict_ms(pipe, X_one, 50)                      # 1 requête (sens du SLA)
    latency_batch_ms_row[name] = _median_predict_ms(pipe, X_test, 20) / len(X_test)  # débit/bien (lot, indicatif)

_hr = results_df.set_index('model')['holdout_rmsle']
print("Latence d'inférence mesurée — SLA Canvas = 5000 ms / soumission (1 bien) :")
print(f"  {'modèle':16} {'req (ms/bien)':>14} {'×sous SLA':>11} {'lot (ms/bien)':>14}   RMSLE")
for name in sorted(FINALISTS, key=lambda m: _hr[m]):
    print(f"  {name:16} {latency_req_ms[name]:>14.3f} {5000/latency_req_ms[name]:>10,.0f}× "
          f"{latency_batch_ms_row[name]:>14.3f}   {_hr[name]:.4f}")

Latence d'inférence mesurée — SLA Canvas = 5000 ms / soumission (1 bien) :
  modèle            req (ms/bien)   ×sous SLA  lot (ms/bien)   RMSLE
  Stacking                 40.071        125×          0.162   0.1122
  GradientBoosting          8.885        563×          0.040   0.1154
  Lasso                    10.374        482×          0.039   0.1165
  CatBoost                 15.587        321×          0.054   0.1175
  XGBoost_tuned            17.946        279×          0.078   0.1193


**Note coût (qualitative — chiffrage détaillé en Phase 6).** Contre-intuitivement, le **compute est quasi gratuit** : même au grain SLA (1 requête = 1 bien, le coût fixe ne s'amortit pas), tous les finalistes prédisent en **quelques dizaines de millisecondes** — du Stacking (~56 ms, ≈ 90× sous le seuil de 5 s) aux modèles uniques (~14–24 ms, ≈ 200–360× sous) — et le réentraînement mensuel ne dure que quelques secondes. Le coût récurrent réel n'est donc **pas** le calcul, mais (1) l'**instance d'API toujours active**, (2) le **monitoring**, et (3) le **workflow de validation par le consultant**. Le « surcoût » du Stacking face à un modèle unique n'est pas le temps de calcul (négligeable) mais sa **surface de monitoring** (4 sous-modèles à surveiller) et sa moindre *debuggabilité*. Le modèle de coût cloud complet est développé en Phase 6 (NB5).

### 5.5.4 Axe métier — interprétabilité, asymétrie et équité

Trois exigences métier ferment l'évaluation :

- **Interprétabilité.** Le Canvas promet « prix + top-3 facteurs » et un *conseil rénovation*. Un Lasso est lisible par ses coefficients ; le Stacking exige une couche **SHAP** (§4.3.7, calculée sur XGBoost comme proxy lisible — le Stacking lui-même n'a pas d'attribution directe).
- **Asymétrie des erreurs.** Posée en §5.5.1 ; non capturée par la RMSLE, elle est suivie *en continu* côté métier en Phase 6 (override, délai de vente).
- **Équité géographique** (contrainte Canvas : pas de *redlining* algorithmique), vérifiée ci-dessous.

**Correction méthodologique (vs première passe).** La première version groupait les résidus sur le seul **holdout** (~292 biens, soit 3–7 par quartier → boxplots peu fiables, ce que la cellule admettait elle-même). On recalcule donc sur des prédictions **out-of-fold** (`cross_val_predict`, 5 folds) couvrant **les 1458 biens** (≈ 5× plus de biens par quartier, médianes bien plus stables). Résidu en **espace log** (`réel − prédit`) pour retirer le biais de gamme de prix ; positif = **sous-estimation**.

In [10]:
from sklearn.model_selection import cross_val_predict

# Prédictions OOF du champion sur les 1458 biens (≈5× plus par quartier que le holdout)
oof_pred = cross_val_predict(build_finalist(champion_name), X, y_log, cv=5, n_jobs=1)
resid_log = y_log.values - oof_pred                      # positif => sous-estimation
neigh = df_feat.loc[X.index, 'Neighborhood'].values
fair = pd.DataFrame({'Neighborhood': neigh, 'resid_log': resid_log})
order = fair.groupby('Neighborhood')['resid_log'].median().sort_values().index

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=fair, x='Neighborhood', y='resid_log', order=order, ax=ax, color='#9b59b6')
ax.axhline(0, color='red', ls='--', lw=1)
ax.set_xlabel('Quartier')
ax.set_ylabel('Résidu log OOF (réel − prédit)')
ax.set_title(f"Équité — résidus OOF (log) par quartier — {champion_name} (1458 biens)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

med = fair.groupby('Neighborhood')['resid_log'].agg(['median', 'count']).sort_values('median')
print(f"OOF RMSLE (champion {champion_name}, 5 folds) : {np.sqrt(np.mean(resid_log**2)):.4f}")
print("\nQuartiers les plus SUR-estimés (médiane < 0) :")
print(med.head(3).to_string())
print("\nQuartiers les plus SOUS-estimés (médiane > 0) :")
print(med.tail(3).to_string())
print(f"\nAmplitude des médianes par quartier : "
      f"[{med['median'].min():+.3f}, {med['median'].max():+.3f}] en log "
      f"(≈ [{100*np.expm1(med['median'].min()):+.1f}%, {100*np.expm1(med['median'].max()):+.1f}%] en prix)")

OOF RMSLE (champion Stacking, 5 folds) : 0.1108

Quartiers les plus SUR-estimés (médiane < 0) :
                median  count
Neighborhood                 
Timber       -0.040482     38
MeadowV      -0.040392     17
SWISU        -0.026346     25

Quartiers les plus SOUS-estimés (médiane > 0) :
                median  count
Neighborhood                 
StoneBr       0.041455     25
IDOTRR        0.043985     37
BrkSide       0.061400     58

Amplitude des médianes par quartier : [-0.040, +0.061] en log (≈ [-4.0%, +6.3%] en prix)


/tmp/ipykernel_234533/617152404.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5.5.5 Matrice de décision multi-critères et choix du champion

On confronte les finalistes sur les **trois axes** à la fois — pas sur la seule RMSLE. La matrice ci-dessous croise : **Math** (RMSLE + IC bootstrap), **Ops** (latence mesurée + surface de monitoring), **Métier** (interprétabilité).

In [11]:
_interp = {'Lasso': 'haute (coefficients directs)',
           'GradientBoosting': 'moyenne (SHAP)',
           'CatBoost': 'moyenne (SHAP)',
           'XGBoost_tuned': 'moyenne (SHAP)',
           'Stacking': 'faible (proxy SHAP via XGBoost)'}
_hr = results_df.set_index('model')['holdout_rmsle']

decision_matrix = pd.DataFrame([{
    'modèle': m,
    'RMSLE [IC95]': f"{_hr[m]:.4f} [{ci[m][0]:.4f}–{ci[m][1]:.4f}]",
    'latence/requête (ms)': round(latency_req_ms[m], 3),
    'surface monitoring': '4 sous-modèles' if m == 'Stacking' else '1 modèle',
    'interprétabilité': _interp[m],
} for m in sorted(FINALISTS, key=lambda m: _hr[m])])

print("Matrice de décision (Math / Ops / Métier) — latence au grain SLA (1 requête) :")
display(decision_matrix)

Matrice de décision (Math / Ops / Métier) — latence au grain SLA (1 requête) :


,modèle,RMSLE [IC95],latence/requête (ms),surface monitoring,interprétabilité
0,Stacking,0.1122 [0.0963–0.1297],40.071,4 sous-modèles,faible (proxy SHAP via XGBoost)
1,GradientBoosting,0.1154 [0.0989–0.1330],8.885,1 modèle,moyenne (SHAP)
2,Lasso,0.1165 [0.1023–0.1308],10.374,1 modèle,haute (coefficients directs)
3,CatBoost,0.1175 [0.0957–0.1411],15.587,1 modèle,moyenne (SHAP)
4,XGBoost_tuned,0.1193 [0.0984–0.1430],17.946,1 modèle,moyenne (SHAP)


**Décision, défendue sur les trois axes.**

- **Mathématique** : le **Stacking** a la meilleure RMSLE holdout (0,1122) mais — l'IC bootstrap (§5.2.1) le dit crûment — son intervalle (≈ [0,096–0,130]) **chevauche celui de presque tout le peloton** (quatorze modèles sur dix-sept). L'avantage est donc *réel mais statistiquement ténu* ; ce qui le distingue est surtout sa **diversité d'erreurs** (meilleure stabilité attendue), confirmée par une RMSLE OOF (0,111) cohérente avec le holdout.
- **Informatique** : sa latence **par requête** (~56 ms, §5.5.3) reste **~90× sous le SLA de 5 s** — confortable, non disqualifiant (un modèle unique serait 2–4× plus rapide encore). Son vrai coût est la **surface de monitoring** (4 sous-modèles), pas le compute (négligeable).
- **Métier** : c'est son point faible — interprétabilité indirecte (proxy SHAP). Acceptable car la décision finale reste **validée par un consultant**, le top-3 facteurs étant fourni via le proxy XGBoost.

**Champion retenu : le Stacking**, en pleine connaissance du quasi ex-æquo : on privilégie la précision et la robustesse pour le label « Prix certifié par Inved AI ».

**Repli chiffré (si l'ops/maintenabilité prime).** Si la direction préfère **un seul artefact** à servir et monitorer, **CatBoost** (≈ 0,117, meilleur booster isolé) ou **XGBoost tuné** (meilleur en CV) ne concèdent que **≈ 0,005 de RMSLE** — dans le bruit de l'IC — tout en divisant par 4 la surface de monitoring et en réduisant encore la latence. Arbitrage **légitime**, pas une dégradation.

**Critère de déploiement (seuil Canvas).** L'équipe s'engage sur **RMSLE ≤ 0,13** sur le holdout de la fenêtre glissante (≈ top-30 % Kaggle) : en-deçà = livrable ; au-delà = investigation + réentraînement (couche 1 du monitoring, Phase 6). Le champion (holdout 0,1122 ; OOF 0,1108) franchit le seuil confortablement.

**Signoff équité.** L'audit **OOF sur les 1458 biens** (§5.5.4) resserre nettement le constat par rapport au holdout : les médianes de résidus par quartier tiennent dans **≈ [−4 %, +6 %]** en prix, sans biais directionnel d'ensemble — pas de *redlining* systématique. Le léger penchant à **sous-estimer BrkSide / IDOTRR (~+5–6 %)** repose désormais sur des effectifs suffisants (n ≈ 40–60) : ce n'est plus un artefact d'échantillon mais un **signal mineur à suivre en production** (couches 1 + 3 du monitoring), pas un motif de rejet. La contrainte d'équité du Canvas est respectée à ce stade.

## 5.6 Transition vers la Phase 6 (déploiement)

Nous disposons désormais des trois conditions de passage en production : un **champion** (Stacking, RMSLE 0,1122), un **critère de déploiement** explicite (≤ 0,13) qu'il franchit, et un **signoff équité** par quartier. La Phase 5 est close ; la suite se déploie sur trois notebooks :

- **NB5 — Déploiement** (`5_deployment.ipynb`) : reconstruit le champion via `results/winning_model.json`, le réentraîne sur tout `X`, génère `submission.csv`, l'enregistre dans le **registre MLflow** (alias `@Production`) et le sert en API.
- **NB6 — Surveillance** (`6_monitoring.ipynb`) : monitoring en trois couches, dérive (PSI), réentraînement et équité continue ; y compris la traçabilité MLflow des 17 modèles.
- **NB7 — Conclusion** (`7_conclusion.ipynb`) : synthèse exécutive du parcours CRISP-ML(Q), à destination de la direction.